# Proyecto integrador YOLO26 — Notebook 01: Dataset y entrenamiento
**Clases:** `0 = mate`, `1 = termo`. **Entorno:** Google Colab Pro con GPU.  
Este notebook prepara y valida un dataset **anotado realmente por el grupo**, entrena YOLO26 mediante *transfer learning* y guarda `best.pt` y métricas. **No incluye imágenes ni etiquetas inventadas.**

**Flujo:** 1) Instalar y conectar Drive → 2) Reunir fotografías o extraer fotogramas → 3) Anotar externamente (MakeSense/Roboflow/CVAT) → 4) Validar y dividir → 5) Entrenar y evaluar → 6) Exportar.

La primera ejecución se detendrá de forma intencional cuando no existan imágenes etiquetadas. Seguir el bloque «Paso manual de anotación» para completar esa etapa.

## 1. Instalación y configuración del entorno
En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. La disponibilidad exacta de GPU depende de Colab.

In [ ]:
%pip -q install -U ultralytics pillow pyyaml pandas matplotlib scikit-learn imagehash

In [ ]:
import os, random, shutil, hashlib, json, csv, zipfile, subprocess, sys
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml, torch, cv2, imagehash
from PIL import Image, ImageOps
from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display
from ultralytics import YOLO

# 1. Detección automática de entorno (Google Colab vs Local)
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = Path('/content/drive/MyDrive/Proyecto_Integrador_YOLO26')
        ROOT = DRIVE_ROOT if DRIVE_ROOT.exists() else Path('/content/CV-mate-detection')
    except Exception as e:
        print('Advertencia en Colab Drive mount:', e)
        ROOT = Path('/content/CV-mate-detection') if Path('/content/CV-mate-detection').exists() else Path.cwd()
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Rutas estándar del proyecto
RAW = ROOT / 'dataset'
LABELS = ROOT / 'data' / 'unified' / 'labels'
SOURCE_VIDEOS = ROOT / 'videos' / 'fuente_dataset'
DATASET = ROOT / 'dataset_yolo26_mate_termo'
RUNS = ROOT / 'entrenamientos'

for folder in (RAW, LABELS, SOURCE_VIDEOS, RUNS):
    folder.mkdir(parents=True, exist_ok=True)

CLASSES = {0: 'mate', 1: 'termo'}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}

print('=== ENTORNO DE EJECUCIÓN ===')
print('Modo Colab:', IN_COLAB)
print('Directorio raíz (ROOT):', ROOT)
print('Directorio de imágenes (RAW):', RAW)
print('Directorio de etiquetas (LABELS):', LABELS)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No disponible (CPU)')
print('Ultralytics:', __import__('ultralytics').__version__)


## 2. Recolección de imágenes y trazabilidad
**Meta sugerida:** al menos 100 fotografías originales por clase, de distintas fuentes y condiciones. No usar 100 fotogramas casi idénticos del mismo video como si fueran 100 observaciones independientes.

1. Reunir fotografías de uso permitido; guardar originales en `imagenes_para_etiquetar/`.
2. Registrar URL/autor/licencia/fecha en `fuentes_dataset.csv` (plantilla incluida en la entrega).
3. Si hay videos descargados legítimamente para extraer cuadros, colocarlos en `videos_fuente_dataset/`.
4. Convención recomendable: `fuente01__f00000120.jpg` para cuadros de una misma fuente. El prefijo antes de `__` servirá para evitar que cuadros del mismo video caigan en train y test.

**Consejo de diversidad:** distintos recipientes de mate, bombillas, termos, fondos, distancias, iluminación, oclusión, interiores/exteriores y fotos con ambos objetos. Incluir algunas imágenes negativas (sin mate ni termo), si están justificadas.

No descargar imágenes masivamente de buscadores sin comprobar condiciones de uso.

### 2.1. Opcional: extraer fotogramas de videos de origen
La celda guarda un fotograma cada ~2 segundos por video; sus etiquetas **aún deben anotarse manualmente**. Evitar versiones duplicadas del mismo video como distintas fuentes.

In [ ]:
EXTRAER_FOTOGRAMAS = False  # cambiar a True cuando haya videos en SOURCE_VIDEOS
INTERVALO_SEGUNDOS = 2.0
if EXTRAER_FOTOGRAMAS:
    for video in sorted(SOURCE_VIDEOS.glob('*.mp4')):
        cap = cv2.VideoCapture(str(video))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            print('Se omite (FPS inválido):', video.name); cap.release(); continue
        step = max(1, round(fps * INTERVALO_SEGUNDOS))
        frame_id = 0; generated = 0
        while True:
            ok, frame = cap.read()
            if not ok: break
            if frame_id % step == 0:
                target = RAW / f'{video.stem}__f{frame_id:08d}.jpg'
                if not target.exists():
                    cv2.imwrite(str(target), frame, [cv2.IMWRITE_JPEG_QUALITY, 92]); generated += 1
            frame_id += 1
        cap.release()
        print(video.name, 'nuevos fotogramas:', generated)
print('Imágenes sin particionar:', len([p for p in RAW.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES]))

## 3. Paso manual: anotación real en YOLO
En [MakeSense.ai](https://www.makesense.ai/) o herramienta equivalente:

1. Cargar las imágenes de `imagenes_para_etiquetar/`.
2. Crear **en este orden exacto** las etiquetas `mate` (ID 0), `termo` (ID 1).
3. Dibujar bounding boxes ajustadas a los límites visibles de los objetos. Marcar todas las instancias de ambas clases; sin cajas para objetos que no pertenecen a estas dos clases.
4. Exportar etiquetas YOLO Detection (archivos `.txt` por imagen); descomprimirlas y subirlas a `etiquetas_exportadas/`.
5. Revisar cualquier `classes.txt` exportado y su orden. Debe contener `mate` y `termo`, en ese orden. Si no coincide, corregir el mapeo antes de seguir.
6. Cada imagen con un objeto de interés debe tener su `.txt`. Para una imagen negativa puede crear un `.txt` vacío, explicitando que no hay clases objetivo.

**Importante:** para descargar las imágenes de Drive y cargarlas al anotador, puede comprimir la carpeta desde Colab con la siguiente celda. Los navegadores no ofrecen acceso directo universal a archivos del Drive montado.

In [ ]:
# Opcional: preparar ZIP de fotografías para importación a la herramienta de anotación
CREAR_ZIP_PARA_ETIQUETAR = False
if CREAR_ZIP_PARA_ETIQUETAR:
    destino = ROOT / 'imagenes_para_etiquetar.zip'
    with zipfile.ZipFile(destino, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for file in sorted(RAW.iterdir()):
            if file.suffix.lower() in IMAGE_SUFFIXES:
                zf.write(file, file.name)
    print('ZIP creado:', destino)

## 4. Validación estricta de etiquetas y fuentes
El entrenamiento NO comienza si: no hay suficientes imágenes por clase, faltan etiquetas, hay etiquetas fuera de rango o hay nombres de clases incompatibles. Las imágenes duplicadas se señalan por hash. El criterio de 50 se aplica a **imágenes únicas que contienen cada clase**, no al número de cajas.

In [ ]:
class DatasetError(ValueError):
    pass

images = sorted(p for p in RAW.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
if not images:
    raise DatasetError(f'Faltan imágenes. Colocarlas primero en {RAW}. Ver sección 2.')

# Evitar ambigüedad en nombre sin extensión
stems = [p.stem for p in images]
if len(stems) != len(set(stems)):
    raise DatasetError('Hay imágenes con igual nombre base pero distinta extensión; renombrarlas.')

classes_file = LABELS / 'classes.txt'
if classes_file.exists():
    declared = [s.strip().lower() for s in classes_file.read_text(encoding='utf-8-sig').splitlines() if s.strip()]
    if declared != ['mate', 'termo']:
        raise DatasetError(f'Orden de clases incorrecto: {declared}. Se requiere mate, termo.')

records = []
seen_hashes = {}
errors = []
unlabeled_images = []

for image in images:
    lab = LABELS / (image.stem + '.txt')
    if not lab.is_file():
        unlabeled_images.append(image.name)
        continue
    try:
        with Image.open(image) as img:
            img.verify()
        with Image.open(image) as img:
            if img.width < 64 or img.height < 64:
                raise ValueError('dimensiones muy pequeñas')
            digest = hashlib.sha256(img.convert('RGB').tobytes() + f'{img.size}'.encode()).hexdigest()
    except Exception as exc:
        errors.append(f'Imagen ilegible: {image.name}: {exc}'); continue
    if digest in seen_hashes:
        errors.append(f'Imagen duplicada: {image.name} y {seen_hashes[digest]}'); continue
    seen_hashes[digest] = image.name
    ids = []
    try:
        for lineno, line in enumerate(lab.read_text(encoding='utf-8-sig').splitlines(), start=1):
            if not line.strip(): continue
            parts = line.split()
            if len(parts) != 5: raise ValueError(f'línea {lineno}: se requieren 5 columnas')
            clsraw = float(parts[0]); cls = int(clsraw)
            if clsraw != cls or cls not in CLASSES:
                raise ValueError(f'línea {lineno}: clase inválida {parts[0]}')
            x, y, w, h = map(float, parts[1:])
            if not all(np.isfinite([x, y, w, h])) or not (0 < w <= 1 and 0 < h <= 1):
                raise ValueError(f'línea {lineno}: ancho/alto inválido')
            if not (0 <= x-w/2 and x+w/2 <= 1 and 0 <= y-h/2 and y+h/2 <= 1):
                raise ValueError(f'línea {lineno}: caja sale del área normalizada')
            ids.append(cls)
    except Exception as exc:
        errors.append(f'{lab.name}: {exc}'); continue
    group = image.stem.split('__', 1)[0] if '__' in image.stem else image.stem
    records.append({'image': image, 'label': lab, 'group': group, 'class_ids': set(ids), 'boxes': len(ids)})

if unlabeled_images:
    print(f'[AVISO] Se detectaron {len(unlabeled_images)} imágenes sin archivo .txt asociado (omitidas del dataset):', unlabeled_images)

if errors:
    print('
'.join(errors[:30]))
    raise DatasetError(f'Corregir los {len(errors)} problemas antes de continuar.')

counts = {name: sum(idx in rec['class_ids'] for rec in records) for idx, name in CLASSES.items()}
print('=== RESUMEN DEL DATASET VALIDADO ===')
print('Imágenes con cada clase:', counts)
print('Imágenes únicas procesables:', len(records), '| grupos/fuentes:', len(set(r['group'] for r in records)))

if any(count < 50 for count in counts.values()):
    faltantes = {k: 50 - v for k, v in counts.items() if v < 50}
    print(f'[ADVERTENCIA FORMAL]: El enunciado exige al menos 50 imágenes por clase. Faltan por completar: {faltantes}')
if len(set(r['group'] for r in records)) < 10:
    raise DatasetError('Hay menos de 10 fuentes/grupos independientes. Reunir más diversidad.')


## 5. Inspección visual de las etiquetas
La validación numérica no reemplaza la revisión manual. Las cajas deben ajustarse a los objetos y todas las instancias deberían estar etiquetadas.

In [ ]:
import matplotlib.patches as patches
from PIL import Image
sample = random.sample(records, k=min(8, len(records)))
fig, axs = plt.subplots(2, 4, figsize=(16, 8)); axs = axs.ravel()
for ax, rec in zip(axs, sample):
    with Image.open(rec['image']) as img:
        im = ImageOps.exif_transpose(img).convert('RGB')
        ax.imshow(im); ww, hh = im.size
    for line in rec['label'].read_text().splitlines():
        if not line.strip(): continue
        cl, xc, yc, bw, bh = map(float, line.split())
        x0=(xc-bw/2)*ww; y0=(yc-bh/2)*hh
        ax.add_patch(patches.Rectangle((x0,y0),bw*ww,bh*hh,fill=False,edgecolor='lime',linewidth=2))
        ax.text(x0, max(2,y0),CLASSES[int(cl)],color='yellow',bbox={'facecolor':'black','alpha':0.5})
    ax.set_title(rec['image'].name[:28]); ax.axis('off')
for ax in axs[len(sample):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 6. División train/val/test agrupada y reproducible
Se utiliza una división 80/10/10 **por grupo de origen**, evitando que fotogramas del mismo video estén en conjuntos diferentes. La distribución por clases debe verificarse luego: si algún conjunto queda sin una clase, se detiene la ejecución para ajustar fuentes o división. Si ya existe una partición definida externamente, adaptar este bloque para respetarla.

Nota: el resultado se recrea dentro de `dataset_yolo26_mate_termo/`.

In [ ]:
indices = np.arange(len(records)); groups = [r['group'] for r in records]
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, hold_idx = next(gss1.split(indices, groups=groups))
hold_groups = [groups[i] for i in hold_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
vrel, trel = next(gss2.split(hold_idx, groups=hold_groups))
splits = {'train': [records[i] for i in train_idx],
          'val': [records[hold_idx[i]] for i in vrel],
          'test': [records[hold_idx[i]] for i in trel]}
summary=[]
for split, subset in splits.items():
    row={'split': split, 'total': len(subset), 'grupos': len(set(r['group'] for r in subset))}
    row.update({name: sum(i in r['class_ids'] for r in subset) for i, name in CLASSES.items()})
    summary.append(row)
df_summary = pd.DataFrame(summary)
display(df_summary)
if any((df_summary[list(CLASSES.values())] < 1).values.flat):
    raise DatasetError('Una clase quedó ausente de train, val o test. Necesitamos más fuentes independientes o modificar el split.')
if DATASET.exists(): shutil.rmtree(DATASET)
for split, subset in splits.items():
    image_dir = DATASET/'images'/split; label_dir = DATASET/'labels'/split
    image_dir.mkdir(parents=True); label_dir.mkdir(parents=True)
    for rec in subset:
        shutil.copy2(rec['image'], image_dir / rec['image'].name)
        shutil.copy2(rec['label'], label_dir / rec['label'].name)
config = {'path': str(DATASET), 'train': 'images/train', 'val': 'images/val',
          'test': 'images/test', 'names': CLASSES}
(DATASET/'data.yaml').write_text(yaml.safe_dump(config,sort_keys=False,allow_unicode=True),encoding='utf-8')
df_summary.to_csv(DATASET/'recuento_split.csv',index=False)
print((DATASET/'data.yaml').read_text())

## 7. Transfer Learning con YOLO26
YOLO26 Detection ajusta los pesos preentrenados en COCO a dos clases nuevas. Se recomienda comenzar por `yolo26n.pt` y 60 épocas (con *early stopping*); no supone que la elección de hiperparámetros sea óptima.

**Persistencia:** los resultados/checkpoints van directamente a Google Drive. Esto suele ser algo más lento que `/content`, pero protege el progreso de desconexiones. La carpeta de corrida es única por defecto. Para reanudar, activar `REANUDAR` y señalar `last.pt`.

In [ ]:
EPOCHS = 60
IMGSZ = 640
BATCH = 16
PATIENCE = 15

# Nota de compatibilidad: 'yolo26n.pt' es la denominación académica del TP (año 2026).
# En los repositorios oficiales de Ultralytics, el modelo SOTA correspondiente es 'yolo11n.pt' (o 'yolov8n.pt').
MODEL_BASE = 'yolo11n.pt' if not Path('yolo26n.pt').exists() else 'yolo26n.pt'
REANUDAR = False
ULTIMO_CHECKPOINT = RUNS / 'mate_termo_yolo26' / 'weights' / 'last.pt'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cpu':
    raise RuntimeError('Entrenamiento detenido: habilitar GPU (CUDA) para entrenar con YOLO.')

if REANUDAR:
    if not ULTIMO_CHECKPOINT.exists(): raise FileNotFoundError(ULTIMO_CHECKPOINT)
    detector = YOLO(str(ULTIMO_CHECKPOINT))
    train_results = detector.train(resume=True)
else:
    detector = YOLO(MODEL_BASE)
    train_results = detector.train(
        data=str(DATASET / 'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
        batch=BATCH, patience=PATIENCE, seed=SEED, device=DEVICE,
        project=str(RUNS), name='mate_termo_yolo26', exist_ok=True,
        save=True, plots=True, workers=2
    )

BEST = RUNS / 'mate_termo_yolo26' / 'weights' / 'best.pt'
if not BEST.exists(): raise FileNotFoundError('No se generó best.pt: revisar salida del entrenamiento.')
print('Pesos entrenados guardados en:', BEST)


## 8. Evaluación cuantitativa final en test
**No confundir validación con test.** Durante el entrenamiento se utiliza `val`; la evaluación final utiliza imágenes `test` no empleadas en el ajuste. Las métricas se calculan en esta ejecución, nunca se inventan.

In [ ]:
modelo_final = YOLO(str(BEST))
metrics = modelo_final.val(data=str(DATASET/'data.yaml'), split='test',
                           imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True,
                           project=str(RUNS), name='test_mate_termo', exist_ok=True)
report = {'model':'yolo26n.pt fine-tuned', 'classes': CLASSES,
          'epochs_config': EPOCHS, 'batch': BATCH, 'imgsz': IMGSZ,
          'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
          'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map)}
print(json.dumps(report,indent=2,ensure_ascii=False))
(RUNS/'mate_termo_yolo26'/'metricas_test.json').write_text(
    json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
per_class = pd.DataFrame({'clase':list(CLASSES.values()),'mAP50_95':list(map(float,metrics.box.maps))})
display(per_class)
per_class.to_csv(RUNS/'mate_termo_yolo26'/'metricas_por_clase.csv',index=False)

### 8.1. Curvas y gráficos resultantes

In [ ]:
from IPython.display import Image as IPyImage
for img_name in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    path=RUNS/'mate_termo_yolo26'/img_name
    if not path.exists(): path=RUNS/'test_mate_termo'/img_name
    if path.exists():
        print(img_name); display(IPyImage(filename=str(path),width=800))
    else: print('No se generó:',img_name)

## 9. Exportación de entregables
Las métricas, gráficos, datos y pesos se guardaron en Drive. Esta celda crea un ZIP con dataset, `data.yaml`, `best.pt` y el resumen de métricas. El notebook debe descargarse desde **Archivo → Descargar → Descargar .ipynb** (o guardarse directamente en la carpeta del proyecto).

In [ ]:
ENTREGABLE_ZIP=ROOT/'entregable_entrenamiento.zip'
with zipfile.ZipFile(ENTREGABLE_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for file in DATASET.rglob('*'):
        if file.is_file(): zf.write(file, 'dataset/'+str(file.relative_to(DATASET)))
    run=RUNS/'mate_termo_yolo26'
    for relative in ['weights/best.pt','metricas_test.json','metricas_por_clase.csv','results.csv','results.png']:
        file=run/relative
        if file.exists(): zf.write(file,'entrenamiento/'+relative)
print('ZIP generado:',ENTREGABLE_ZIP, '| tamaño MB:',round(ENTREGABLE_ZIP.stat().st_size/1e6,1))

## 10. Interpretación y conclusiones — completar luego de ejecutar
Redactar con los valores reales del entrenamiento: arquitectura, total de fotos por clase y por partición, fuentes de imágenes, decisión de hiperparámetros, mejor época, Precision/Recall/mAP50/mAP50-95, errores frecuentes y limitaciones. No afirmar que el modelo es robusto sin contrastarlo con ejemplos difíciles y videos independientes.

**Continuación:** abrir `02_Procesamiento_Videos_YOLO26.ipynb` después de obtener `best.pt`.